# EDA inicial — `data/raw`

Exploración superficial de los CSV en `data/raw/` (tres orígenes: resultados internacionales, tablas de la Copa del Mundo [jfjelstul/worldcup] y el dataset Kaggle FIFA World Cup).

- **Columnas y ejemplos de cada archivo:** [docs/data_raw_structure.md](../docs/data_raw_structure.md)
- **Descarga de datos:** desde la raíz del proyecto, `python -m src.data_acquisition` (ver [README.md](../README.md)).
- **Siguiente paso:** [01_eda_visuals.ipynb](01_eda_visuals.ipynb) trabaja sobre `data/processed/`.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

RAW_DIR = Path("../data/raw")


def preview_csv(path: Path, n: int = 3) -> None:
    """Muestra shape, nombres de columnas y las primeras filas de un CSV bajo data/raw."""
    rel = path.relative_to(RAW_DIR).as_posix()
    print(f"\n--- {rel} ---")
    if not path.is_file():
        print("(archivo no encontrado)")
        return
    df_local = pd.read_csv(path)
    print("shape:", df_local.shape)
    print("columnas:", df_local.columns.tolist())
    display(df_local.head(n))


if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f"No existe la carpeta {RAW_DIR.resolve()!s}. "
        "Desde la raíz del repositorio ejecuta:  python -m src.data_acquisition"
    )
print(f"OK: usando datos en {RAW_DIR.resolve()!s}")


OK: usando datos en C:\Users\echoe\Desktop\lab01\data\raw


## Inventario de CSV

Se listan todos los `.csv` bajo `data/raw/` con número de filas (conteo por líneas, sin cargar todo el archivo), número de columnas y lista de columnas truncada si hay muchas. Las tablas auxiliares de `worldcup_data/` aparecen aquí; el detalle de columnas está en la documentación enlazada arriba.


In [2]:
def count_csv_rows(path: Path) -> int:
    with path.open(encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in f) - 1, 0)


paths = sorted(RAW_DIR.rglob("*.csv"))
rows = []
for p in paths:
    header = pd.read_csv(p, nrows=0)
    cols = list(header.columns)
    if len(cols) <= 20:
        col_str = ", ".join(cols)
    else:
        col_str = ", ".join(cols[:20]) + ", ..."
    rel = p.relative_to(RAW_DIR).as_posix()
    rows.append(
        {
            "ruta_relativa": rel,
            "grupo": rel.split("/")[0] if "/" in rel else ".",
            "filas": count_csv_rows(p),
            "n_columnas": len(cols),
            "columnas": col_str,
        }
    )

inv = pd.DataFrame(rows).sort_values(["grupo", "ruta_relativa"]).reset_index(drop=True)
print(f"Total archivos CSV: {len(inv)}")
display(inv.drop(columns=["grupo"]))

print("\nPor carpeta (agrupación rápida):")
for g, sub in inv.groupby("grupo", sort=False):
    print("\n### (raíz)" if g == "." else f"\n### {g}/")
    display(sub[["ruta_relativa", "filas", "n_columnas"]].reset_index(drop=True))


Total archivos CSV: 31


,ruta_relativa,filas,n_columnas,columnas
0,international_results.csv,49329,9,"date, home_team, away_team, home_score, away_s..."
1,fifa-world-cup/WorldCupMatches.csv,4572,20,"Year, Datetime, Stage, Stadium, City, Home Tea..."
2,fifa-world-cup/WorldCupPlayers.csv,37784,9,"RoundID, MatchID, Team Initials, Coach Name, L..."
3,fifa-world-cup/WorldCups.csv,20,10,"Year, Country, Winner, Runners-Up, Third, Four..."
4,worldcup_data/award_winners.csv,200,12,"key_id, tournament_id, tournament_name, award_..."
5,worldcup_data/awards.csv,8,5,"key_id, award_id, award_name, award_descriptio..."
6,worldcup_data/bookings.csv,3178,26,"key_id, booking_id, tournament_id, tournament_..."
7,worldcup_data/confederations.csv,6,5,"key_id, confederation_id, confederation_name, ..."
8,worldcup_data/goals.csv,3637,27,"key_id, goal_id, tournament_id, tournament_nam..."
9,worldcup_data/group_standings.csv,626,19,"key_id, tournament_id, tournament_name, stage_..."



Por carpeta (agrupación rápida):

### (raíz)


,ruta_relativa,filas,n_columnas
0,international_results.csv,49329,9



### fifa-world-cup/


,ruta_relativa,filas,n_columnas
0,fifa-world-cup/WorldCupMatches.csv,4572,20
1,fifa-world-cup/WorldCupPlayers.csv,37784,9
2,fifa-world-cup/WorldCups.csv,20,10



### worldcup_data/


,ruta_relativa,filas,n_columnas
0,worldcup_data/award_winners.csv,200,12
1,worldcup_data/awards.csv,8,5
2,worldcup_data/bookings.csv,3178,26
3,worldcup_data/confederations.csv,6,5
4,worldcup_data/goals.csv,3637,27
5,worldcup_data/group_standings.csv,626,19
6,worldcup_data/groups.csv,159,7
7,worldcup_data/host_countries.csv,31,7
8,worldcup_data/manager_appearances.csv,2538,17
9,worldcup_data/manager_appointments.csv,637,10


## `international_results.csv`

Partidos internacionales históricos (fuente [martj42/international_results](https://github.com/martj42/international_results)). Las columnas `home_score` y `away_score` son los goles del local y del visitante.


In [3]:
intl_path = RAW_DIR / "international_results.csv"
df = pd.read_csv(intl_path, parse_dates=["date"])
df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 49329 entries, 0 to 49328
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49329 non-null  datetime64[us]
 1   home_team   49329 non-null  str           
 2   away_team   49329 non-null  str           
 3   home_score  49257 non-null  float64       
 4   away_score  49257 non-null  float64       
 5   tournament  49329 non-null  str           
 6   city        49329 non-null  str           
 7   country     49329 non-null  str           
 8   neutral     49329 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(2), str(5)
memory usage: 3.1 MB


In [5]:
df.isnull().sum()


date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [6]:
print("Rango de fechas:", df["date"].min(), "→", df["date"].max())
print("\nTop 10 torneos por número de partidos:")
display(df["tournament"].value_counts().head(10))


Rango de fechas: 1872-11-30 00:00:00 → 2026-06-27 00:00:00

Top 10 torneos por número de partidos:


tournament
Friendly                                18257
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64

## `fifa-world-cup/` (Kaggle)

Tres CSV del dataset [abecklas/fifa-world-cup](https://www.kaggle.com/datasets/abecklas/fifa-world-cup). Se usan en preprocesado junto con los otros orígenes.


In [7]:
for name in ("WorldCupMatches.csv", "WorldCupPlayers.csv", "WorldCups.csv"):
    preview_csv(RAW_DIR / "fifa-world-cup" / name)



--- fifa-world-cup/WorldCupMatches.csv ---
shape: (4572, 20)
columnas: ['Year', 'Datetime', 'Stage', 'Stadium', 'City', 'Home Team Name', 'Home Team Goals', 'Away Team Goals', 'Away Team Name', 'Win conditions', 'Attendance', 'Half-time Home Goals', 'Half-time Away Goals', 'Referee', 'Assistant 1', 'Assistant 2', 'RoundID', 'MatchID', 'Home Team Initials', 'Away Team Initials']


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930.0,13 Jul 1930 - 15:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930.0,13 Jul 1930 - 15:00,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,,18346.0,2.0,0.0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201.0,1090.0,USA,BEL
2,1930.0,14 Jul 1930 - 12:45,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,,24059.0,2.0,0.0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201.0,1093.0,YUG,BRA



--- fifa-world-cup/WorldCupPlayers.csv ---
shape: (37784, 9)
columnas: ['RoundID', 'MatchID', 'Team Initials', 'Coach Name', 'Line-up', 'Shirt Number', 'Player Name', 'Position', 'Event']


,RoundID,MatchID,Team Initials,Coach Name,Line-up,Shirt Number,Player Name,Position,Event
0,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Alex THEPOT,GK,NaN
1,201,1096,MEX,LUQUE Juan (MEX),S,0,Oscar BONFIGLIO,GK,NaN
2,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Marcel LANGILLER,NaN,G40'



--- fifa-world-cup/WorldCups.csv ---
shape: (20, 10)
columnas: ['Year', 'Country', 'Winner', 'Runners-Up', 'Third', 'Fourth', 'GoalsScored', 'QualifiedTeams', 'MatchesPlayed', 'Attendance']


,Year,Country,Winner,Runners-Up,Third,Fourth,GoalsScored,QualifiedTeams,MatchesPlayed,Attendance
0,1930,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590.549
1,1934,Italy,Italy,Czechoslovakia,Germany,Austria,70,16,17,363.000
2,1938,France,Italy,Hungary,Brazil,Sweden,84,15,18,375.700


## `worldcup_data/` — archivos clave del pipeline

Vista previa de tablas que consume el código en `src/preprocessing.py` y `src/feature_engineering.py`. El inventario de la primera sección lista el resto de CSV (tarjetas, goles, árbitros, etc.).


In [8]:
for name in (
    "matches.csv",
    "tournaments.csv",
    "teams.csv",
    "squads.csv",
    "players.csv",
):
    preview_csv(RAW_DIR / "worldcup_data" / name)



--- worldcup_data/matches.csv ---
shape: (1248, 37)
columnas: ['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name', 'stage_name', 'group_name', 'group_stage', 'knockout_stage', 'replayed', 'replay', 'match_date', 'match_time', 'stadium_id', 'stadium_name', 'city_name', 'country_name', 'home_team_id', 'home_team_name', 'home_team_code', 'away_team_id', 'away_team_name', 'away_team_code', 'score', 'home_team_score', 'away_team_score', 'home_team_score_margin', 'away_team_score_margin', 'extra_time', 'penalty_shootout', 'score_penalties', 'home_team_score_penalties', 'away_team_score_penalties', 'result', 'home_team_win', 'away_team_win', 'draw']


,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,...,away_team_score_margin,extra_time,penalty_shootout,score_penalties,home_team_score_penalties,away_team_score_penalties,result,home_team_win,away_team_win,draw
0,1,WC-1930,1930 FIFA Men's World Cup,M-1930-01,France vs Mexico,group stage,Group 1,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
1,2,WC-1930,1930 FIFA Men's World Cup,M-1930-02,United States vs Belgium,group stage,Group 4,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
2,3,WC-1930,1930 FIFA Men's World Cup,M-1930-03,Yugoslavia vs Brazil,group stage,Group 2,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0



--- worldcup_data/tournaments.csv ---
shape: (30, 18)
columnas: ['key_id', 'tournament_id', 'tournament_name', 'year', 'start_date', 'end_date', 'host_country', 'winner', 'host_won', 'count_teams', 'group_stage', 'second_group_stage', 'final_round', 'round_of_16', 'quarter_finals', 'semi_finals', 'third_place_match', 'final']


,key_id,tournament_id,tournament_name,year,start_date,end_date,host_country,winner,host_won,count_teams,group_stage,second_group_stage,final_round,round_of_16,quarter_finals,semi_finals,third_place_match,final
0,1,WC-1930,1930 FIFA Men's World Cup,1930,1930-07-13,1930-07-30,Uruguay,Uruguay,1,13,1,0,0,0,0,1,0,1
1,2,WC-1934,1934 FIFA Men's World Cup,1934,1934-05-27,1934-06-10,Italy,Italy,1,16,0,0,0,1,1,1,1,1
2,3,WC-1938,1938 FIFA Men's World Cup,1938,1938-06-04,1938-06-19,France,Italy,0,15,0,0,0,1,1,1,1,1



--- worldcup_data/teams.csv ---
shape: (88, 14)
columnas: ['key_id', 'team_id', 'team_name', 'team_code', 'mens_team', 'womens_team', 'federation_name', 'region_name', 'confederation_id', 'confederation_name', 'confederation_code', 'mens_team_wikipedia_link', 'womens_team_wikipedia_link', 'federation_wikipedia_link']


,key_id,team_id,team_name,team_code,mens_team,womens_team,federation_name,region_name,confederation_id,confederation_name,confederation_code,mens_team_wikipedia_link,womens_team_wikipedia_link,federation_wikipedia_link
0,1,T-01,Algeria,DZA,1,0,Algerian Football Federation,Africa,CF-2,Confederation of African Football,CAF,https://en.wikipedia.org/wiki/Algeria_national...,not applicable,https://en.wikipedia.org/wiki/Algerian_Footbal...
1,2,T-02,Angola,AGO,1,0,Angolan Football Federation,Africa,CF-2,Confederation of African Football,CAF,https://en.wikipedia.org/wiki/Angola_national_...,not applicable,https://en.wikipedia.org/wiki/Angolan_Football...
2,3,T-03,Argentina,ARG,1,1,Argentine Football Association,South America,CF-4,South American Football Confederation,CONMEBOL,https://en.wikipedia.org/wiki/Argentina_nation...,https://en.wikipedia.org/wiki/Argentina_women'...,https://en.wikipedia.org/wiki/Argentine_Footba...



--- worldcup_data/squads.csv ---
shape: (13843, 12)
columnas: ['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name', 'team_code', 'player_id', 'family_name', 'given_name', 'shirt_number', 'position_name', 'position_code']


,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,position_name,position_code
0,1,WC-1930,1930 FIFA Men's World Cup,T-03,Argentina,ARG,P-69244,Bossio,Ángel,0,goal keeper,GK
1,2,WC-1930,1930 FIFA Men's World Cup,T-03,Argentina,ARG,P-23160,Botasso,Juan,0,goal keeper,GK
2,3,WC-1930,1930 FIFA Men's World Cup,T-03,Argentina,ARG,P-99230,Cherro,Roberto,0,forward,FW



--- worldcup_data/players.csv ---
shape: (10401, 13)
columnas: ['key_id', 'player_id', 'family_name', 'given_name', 'birth_date', 'female', 'goal_keeper', 'defender', 'midfielder', 'forward', 'count_tournaments', 'list_tournaments', 'player_wikipedia_link']


,key_id,player_id,family_name,given_name,birth_date,female,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,player_wikipedia_link
0,1,P-35894,A'Court,Alan,1934-09-30,0,0,0,0,1,1,1958,https://en.wikipedia.org/wiki/Alan_A%27Court
1,2,P-29915,Aarønes,Ann Kristin,1973-01-19,1,0,0,1,1,2,"1995, 1999",https://en.wikipedia.org/wiki/Ann_Kristin_Aar%...
2,3,P-03484,Aaronson,Brenden,2000-10-22,0,0,0,0,1,1,2022,https://en.wikipedia.org/wiki/Brenden_Aaronson


## Cierre

- **Fase 1 (`preprocessing`):** combina `international_results.csv`, `worldcup_data/matches.csv`, `worldcup_data/teams.csv` y `fifa-world-cup/WorldCupMatches.csv` para construir el dataset maestro.
- **Fase 2 (`feature_engineering`):** añade señales a partir de `worldcup_data/squads.csv`, `players.csv` y `tournaments.csv`, entre otros.

Con los datos revisados, el siguiente paso es ejecutar desde la raíz: `python -m src.preprocessing`.
